# Tools, Tool Calling and Agents

**Title:** Tools, Tool Calling and Agents  
**Difficulty:** Advanced to Expert  
**Notebook:** 06 of 07  

---

> *"A chain follows a recipe. An agent reads the recipe, decides which tools to use, and improvises when things go wrong."*

In this notebook you will learn how to give an LLM the ability to **call external functions**, and how to build an **agent** that autonomously decides which tool to use and when.

## Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** the difference between chains, tools, and agents
2. **Create** tools from Python functions using the `@tool` decorator
3. **Understand** how tool schemas are generated and passed to models
4. **Use** `bind_tools()` to let a model call tools
5. **Read** `AIMessage.tool_calls` to see what the model decided
6. **Build** a complete agent that autonomously selects and executes tools
7. **Implement** Data Science tools (mean, median, std, correlation, summary)
8. **Compare** chain vs agent architectures and know when to use each
9. **Handle** tool errors gracefully
10. **Apply** responsible AI and security practices for tool-using agents

## Prerequisites

| Concept | Notebook |
|---------|----------|
| LangChain basics & LCEL chains | 03 |
| Chat models & prompts | 02 |
| Python functions & type hints | Background |
| Basic statistics (mean, std) | Background |

> **Time estimate:** 90-120 minutes

## Setup

In [ ]:
# Standard imports
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

print("All imports loaded successfully!")

In [ ]:
# Verify API key (if using OpenAI)
api_key = os.getenv("OPENAI_API_KEY", "")
if api_key:
    print(f"OPENAI_API_KEY is set (starts with '{api_key[:8]}...')")
else:
    print("OPENAI_API_KEY not found -- Ollama examples will still work!")

## 1. The Problem with Fixed Chains

In notebooks 03 and 05 we built **chains** -- fixed sequences of steps:

```
Input -> Prompt -> Model -> Output
```

Chains are powerful but **rigid**. They always execute the same steps in the same order.

**What if the user asks different kinds of questions?**

| User asks | Chain needs |
|-----------|-------------|
| "What is the mean of [1,2,3]?" | A calculation chain |
| "Summarize this dataset" | A summarization chain |
| "What evaluation metric should I use?" | A knowledge chain |

A chain can only do **one** of these. You would need **separate chains** for each task.

> **The agent pattern solves this:** give the LLM access to *many* tools and let *it* decide which one to use.

## 2. Chain vs Agent -- Key Difference

```mermaid
graph TD
    subgraph Chain["Chain -- Fixed Workflow"]
        C1[Input] --> C2[Prompt]
        C2 --> C3[Model]
        C3 --> C4[Output]
    end
    
    subgraph Agent["Agent -- Dynamic Workflow"]
        A1[User Request] --> A2{LLM Decides}
        A2 -->|Tool needed| A3[Execute Tool]
        A3 --> A4[Tool Result]
        A4 --> A2
        A2 -->|No tool needed| A5[Final Answer]
    end
```

| Property | Chain | Agent |
|----------|-------|-------|
| **Workflow** | Fixed, predetermined | Dynamic, decided by LLM |
| **Steps** | Always same order | Varies per request |
| **Tool selection** | Hardcoded | LLM chooses autonomously |
| **Loop** | No (runs once) | Yes (tool -> result -> decide again) |
| **Transparency** | High -- you know the flow | Lower -- LLM decides |
| **Cost** | Predictable | Variable (multiple LLM calls possible) |
| **Best for** | Known, repeatable tasks | Flexible, multi-capability tasks |

## 3. What is a Tool?

A **tool** is a Python function that the LLM can call. The LLM sees:

1. **Tool name** -- what the function does (e.g., `calculate_mean`)
2. **Description** -- from the docstring (tells the LLM *when* to use it)
3. **Input schema** -- parameter names, types, and descriptions
4. **Output** -- the return value after execution

```
User: "What is the mean of 10, 20, 30?"

  LLM sees tools: [calculate_mean, calculate_median, ...]
  LLM decides: "I should call calculate_mean"
  LLM outputs: tool_call(name="calculate_mean", args={...})

  Your code executes: calculate_mean([10, 20, 30]) -> 20.0

  LLM receives: 20.0
  LLM responds: "The mean of 10, 20, 30 is 20.0"
```

> **Key insight:** The LLM *never* executes code directly. It *requests* a tool call, and your code runs the function.

## 4. Creating Tools with `@tool`

The `@tool` decorator from `langchain_core.tools` turns any Python function into a LangChain tool.

**Rules:**
1. The **function name** becomes the tool name
2. The **docstring** becomes the description (be thorough!)
3. **Type hints** define the input schema
4. The **return value** is sent back to the LLM

In [ ]:
from langchain_core.tools import tool
import numpy as np
import pandas as pd


@tool
def calculate_mean(values: list[float]) -> float:
    """Calculate the arithmetic mean (average) of a list of numbers.

    Use this tool when the user asks for the average, mean, or
    arithmetic center of a set of numbers.

    Args:
        values: A list of numerical values to average.

    Returns:
        The arithmetic mean as a float.
    """
    return float(np.mean(values))


@tool
def calculate_median(values: list[float]) -> float:
    """Calculate the median (middle value) of a list of numbers.

    Use this tool when the user asks for the median or middle value
    of a dataset.

    Args:
        values: A list of numerical values.

    Returns:
        The median value as a float.
    """
    return float(np.median(values))


@tool
def calculate_std(values: list[float]) -> float:
    """Calculate the standard deviation of a list of numbers.

    Use this tool when the user asks for the standard deviation,
    spread, or variability of a dataset.

    Args:
        values: A list of numerical values.

    Returns:
        The sample standard deviation as a float.
    """
    return float(np.std(values, ddof=1))


@tool
def calculate_correlation(x_values: list[float], y_values: list[float]) -> float:
    """Calculate the Pearson correlation coefficient between two lists of numbers.

    Use this tool when the user asks about the relationship,
    correlation, or association between two variables.

    Args:
        x_values: First list of numerical values.
        y_values: Second list of numerical values (same length as x_values).

    Returns:
        The Pearson correlation coefficient as a float (between -1 and 1).
    """
    if len(x_values) != len(y_values):
        raise ValueError("Both lists must have the same length")
    return float(np.corrcoef(x_values, y_values)[0, 1])


@tool
def dataset_summary(csv_content: str) -> str:
    """Generate a statistical summary of a dataset provided as CSV content.

    Use this tool when the user wants a summary, overview, or
    descriptive statistics of a tabular dataset.

    Args:
        csv_content: The dataset as a CSV-formatted string.

    Returns:
        A formatted string with shape, data types, descriptive
        statistics, and missing value counts.
    """
    from io import StringIO
    df = pd.read_csv(StringIO(csv_content))
    
    lines = [
        f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns",
        f"Columns: {list(df.columns)}",
        f"Data types:\n{df.dtypes.to_string()}",
        f"Descriptive statistics:\n{df.describe().to_string()}",
        f"Missing values:\n{df.isnull().sum().to_string()}",
    ]
    return "\n".join(lines)


print("5 tools created:")
print("  1. calculate_mean")
print("  2. calculate_median")
print("  3. calculate_std")
print("  4. calculate_correlation")
print("  5. dataset_summary")

## 5. Understanding Tool Schemas

When you create a tool with `@tool`, LangChain automatically generates a **JSON schema** that describes the tool to the LLM. Let's inspect it:

In [ ]:
# Inspect the tool schema that the LLM will see
for t in [calculate_mean, calculate_median, calculate_std, calculate_correlation]:
    schema = t.get_input_schema()
    print(f"Tool: {schema['title']}")
    print(f"  Description: {schema['description'][:80]}...")
    print(f"  Properties: {list(schema['properties'].keys())}")
    print()

In [ ]:
# The schema is what gets sent to the model
# Here is what the LLM actually receives (conceptually):
import json
schema = calculate_mean.get_input_schema()
print(json.dumps(schema, indent=2))

### What happened?

- The function's **name** -> tool name in the schema
- The **docstring** -> `description` field  
- The **type hints** (`list[float]`) -> JSON Schema `type: array, items: {type: number}`
- The **argument names** -> property keys

> This schema is what the model uses to decide which tool to call and what arguments to pass.

## 6. Tool Calling -- Letting the Model Decide

**Tool calling** is the mechanism by which an LLM outputs a request to call a specific tool. Here's the flow:

```
User question
    |
LLM receives question + tool schemas
    |
LLM decides: "I should call calculate_mean with values=[10, 20, 30]"
    |
Model outputs: AIMessage with tool_calls=[{name: ..., args: ...}]
    |
YOUR code executes: calculate_mean([10, 20, 30]) -> 20.0
    |
Result sent back to LLM
    |
LLM formulates final answer
```

Let's see this in action:

In [ ]:
# Step 1: Create a model and bind tools to it
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model_with_tools = model.bind_tools([calculate_mean, calculate_median, calculate_std])

# Step 2: Ask a question
response = model_with_tools.invoke([
    SystemMessage(content="You are a helpful data science assistant. Use tools when appropriate."),
    HumanMessage(content="What is the mean of 10, 20, 30, 40, 50?")
])

# Step 3: Inspect the response
print("Response type:", type(response).__name__)
print("Content:", response.content)
print("Tool calls:", response.tool_calls)

### What happened?

1. We created a `ChatOpenAI` model
2. We called `bind_tools([...])` to attach our tool schemas
3. The model received our question and the tool definitions
4. Instead of answering directly, it decided to call `calculate_mean`
5. The `tool_calls` attribute shows exactly what the model requested

> **The model did NOT execute any code.** It only *requested* that we call `calculate_mean` with the given arguments.

In [ ]:
# Let's execute the tool call ourselves
tool_call = response.tool_calls[0]
print(f"Tool name: {tool_call['name']}")
print(f"Arguments: {tool_call['args']}")

# Execute the tool
if tool_call["name"] == "calculate_mean":
    result = calculate_mean.invoke(tool_call["args"])
    print(f"Result: {result}")
elif tool_call["name"] == "calculate_median":
    result = calculate_median.invoke(tool_call["args"])
    print(f"Result: {result}")
elif tool_call["name"] == "calculate_std":
    result = calculate_std.invoke(tool_call["args"])
    print(f"Result: {result}")

In [ ]:
# Experiment yourself! Try these questions:
test_questions = [
    "What is the median of 5, 15, 25, 35?",
    "Calculate the standard deviation of 100, 200, 300",
    "What is the mean of 3.14, 2.71, 1.41?",
]

for q in test_questions:
    response = model_with_tools.invoke([
        SystemMessage(content="You are a data science assistant. Use tools when appropriate."),
        HumanMessage(content=q)
    ])
    print(f"Q: {q}")
    if response.tool_calls:
        tc = response.tool_calls[0]
        print(f"   -> Tool: {tc['name']}, Args: {tc['args']}")
    else:
        print(f"   -> Direct answer: {response.content[:100]}")
    print()

## 7. Handling Multiple Tool Calls

A model can request **multiple tool calls** in a single response:

In [ ]:
response = model_with_tools.invoke([
    SystemMessage(content="You are a data science assistant. Use tools when appropriate."),
    HumanMessage(content="For the data [2, 4, 6, 8, 10, 12], calculate the mean AND the standard deviation.")
])

print(f"Number of tool calls: {len(response.tool_calls)}")
for tc in response.tool_calls:
    print(f"  -> {tc['name']}({tc['args']})")
    
# Execute all tool calls
results = {}
for tc in response.tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    
    # Map tool name to function
    tool_map = {
        "calculate_mean": calculate_mean,
        "calculate_median": calculate_median,
        "calculate_std": calculate_std,
    }
    if tool_name in tool_map:
        result = tool_map[tool_name].invoke(tool_args)
        results[tool_name] = result
        print(f"  Result: {tool_name} = {result}")

## 8. The Agent Loop

An **agent** automates the entire process of:
1. Sending a request to the model
2. Checking if the model wants to call tools
3. Executing those tools
4. Sending results back
5. Repeating until the model produces a final answer

```mermaid
graph TD
    A[User Question] --> B[Send to LLM with tools]
    B --> C{LLM Response}
    C -->|tool_calls| D[Execute Tool]
    D --> E[Send result back to LLM]
    E --> B
    C -->|No tool calls| F[Return Final Answer]
    
    style A fill:#e1f5fe
    style D fill:#fff3e0
    style F fill:#e8f5e9
```

This is the **ReAct pattern** (Reasoning + Acting): the model thinks, acts, observes, and repeats.

## 9. Building an Agent with `create_tool_calling_agent`

LangChain provides `create_tool_calling_agent` which creates a complete agent that:
- Receives user input
- Calls tools as needed
- Returns a final answer

We combine it with a prompt and `AgentExecutor`:

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Define the agent prompt
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Data Science calculator assistant. "
     "You have access to tools for computing statistical measures. "
     "Always use tools for calculations -- never compute manually. "
     "When you get tool results, provide a clear explanation."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),  # Required for tool calling agents
])

# Create the agent
agent = create_tool_calling_agent(
    model,
    [calculate_mean, calculate_median, calculate_std, calculate_correlation],
    agent_prompt,
)

# Create the executor (runs the agent loop)
executor = AgentExecutor(
    agent=agent,
    tools=[calculate_mean, calculate_median, calculate_std, calculate_correlation],
    verbose=True,
)

print("Agent created!")

In [ ]:
# Test the agent
result = executor.invoke({
    "input": "What is the mean and standard deviation of [10, 20, 30, 40, 50]?"
})

print("\n" + "=" * 60)
print(f"Final answer: {result['output']}")

### What happened?

1. `create_tool_calling_agent` wired together the **model**, **tools**, and **prompt**
2. `AgentExecutor` runs the **agent loop**:
   - Sends user question to the LLM
   - LLM decides to call `calculate_mean` -> executor runs it
   - LLM decides to call `calculate_std` -> executor runs it  
   - LLM receives both results -> generates final answer
3. The `verbose=True` flag shows us the internal steps

## 10. Building an Agent with `create_agent` (LangChain 1.0)

LangChain 1.0 introduced a simpler `create_agent` function that eliminates boilerplate:

In [ ]:
from langchain.agents import create_agent

# The modern approach -- much simpler!
ds_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[calculate_mean, calculate_median, calculate_std, calculate_correlation],
    system_prompt="You are a Data Science calculator assistant. "
     "You have access to tools for computing statistical measures. "
     "Always use tools for calculations -- never compute manually.",
)

# Invoke directly -- no AgentExecutor needed!
result = ds_agent.invoke({"role": "user", "content": "What is the median of 15, 25, 35, 45, 55?"})

print(f"Agent response: {result}")

### `create_tool_calling_agent` vs `create_agent`

| Feature | `create_tool_calling_agent` | `create_agent` |
|---------|---------------------------|----------------|
| **Package** | `langchain.agents` | `langchain.agents` |
| **Requires** | `AgentExecutor` wrapper | Direct `.invoke()` |
| **Prompt format** | Custom with `agent_scratchpad` | Simple system prompt string |
| **Model** | Pass model object | Pass `"provider:model"` string or object |
| **Flexibility** | Full control over executor | Simpler, built-in loop |
| **Best for** | Learning internals, custom control | Quick prototyping, standard patterns |

> Both approaches work. `create_agent` is simpler for beginners. `create_tool_calling_agent` gives more control.

## 11. Local Ollama Implementation

Ollama models with tool-calling support (like `llama3.2`, `qwen2.5`) work similarly:

### Prerequisites

```bash
# Make sure Ollama is running
ollama --version

# Pull a model with tool-calling support
ollama pull llama3.2

# Verify the model is available
ollama list
```

In [ ]:
# Local Ollama agent
local_model = ChatOllama(model="llama3.2", temperature=0)

local_agent = create_tool_calling_agent(
    local_model,
    [calculate_mean, calculate_median, calculate_std, calculate_correlation],
    agent_prompt,
)

local_executor = AgentExecutor(
    agent=local_agent,
    tools=[calculate_mean, calculate_median, calculate_std, calculate_correlation],
    verbose=True,
)

print("Local Ollama agent created!")
print("This runs entirely on your machine -- no API key needed!")

In [ ]:
# Test the local agent
result = local_executor.invoke({
    "input": "What is the mean of 5, 10, 15, 20, 25?"
})

print("\n" + "=" * 60)
print(f"Local answer: {result['output']}")

### API vs Local Comparison

| Feature | OpenAI API | Local Ollama |
|---------|-----------|--------------|
| **Tool calling quality** | Excellent | Good (improving) |
| **Speed** | Depends on network | Depends on hardware |
| **Cost** | Per-token pricing | Free after download |
| **Privacy** | Data sent to API | Stays on your machine |
| **Model quality** | GPT-4o: very high | Llama 3.2: good |
| **Reliability** | Very reliable | Tool calling may vary by model |

> **Tip for students:** Start with the API for reliability. Switch to Ollama once your code works.

## 12. Complete Data Science Agent

Let's build a comprehensive agent with a **dataset summary tool** and test it with real data:

In [ ]:
# Create a sample dataset
sample_data = pd.DataFrame({
    "age": [25, 30, 35, 40, 45, 50, 55, 60, 28, 33],
    "income": [30000, 45000, 55000, 65000, 80000, 95000, 110000, 120000, 35000, 50000],
    "score": [72, 85, 78, 90, 88, 95, 92, 97, 75, 82]
})

# Convert to CSV for the tool
csv_string = sample_data.to_csv(index=False)
print("Sample dataset:")
print(sample_data)
print(f"\nCSV length: {len(csv_string)} characters")

In [ ]:
# Create a comprehensive agent with all tools
all_tools = [calculate_mean, calculate_median, calculate_std, calculate_correlation, dataset_summary]

comprehensive_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert Data Science assistant with access to statistical tools. "
     "When answering questions: "
     "Always use the appropriate statistical tool for calculations. "
     "For dataset analysis, use the dataset_summary tool. "
     "Explain results in plain language. "
     "If asked about relationships, use the correlation tool with both variables."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

comprehensive_agent = create_tool_calling_agent(model, all_tools, comprehensive_prompt)
comprehensive_executor = AgentExecutor(agent=comprehensive_agent, tools=all_tools, verbose=True)

print("Comprehensive Data Science agent ready!")

In [ ]:
# Test 1: Dataset analysis
result = comprehensive_executor.invoke({
    "input": f"Summarize this dataset:\n\n{csv_string}"
})

print("\n" + "=" * 60)
print(f"Analysis: {result['output'][:500]}")

In [ ]:
# Test 2: Specific calculation
result = comprehensive_executor.invoke({
    "input": "What is the correlation between age and income? Is it strong?"
})

print("\n" + "=" * 60)
print(f"Answer: {result['output'][:500]}")

## 13. API vs Ollama -- Side by Side

Here is the same agent built with both approaches:

In [ ]:
# ============================================
# API VERSION (OpenAI)
# ============================================
def create_api_agent():
    "Create an agent using the OpenAI API."
    api_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return create_tool_calling_agent(api_model, all_tools, comprehensive_prompt)

# ============================================
# LOCAL VERSION (Ollama)
# ============================================
def create_local_agent():
    "Create an agent using local Ollama."
    local_model = ChatOllama(model="llama3.2", temperature=0)
    return create_tool_calling_agent(local_model, all_tools, comprehensive_prompt)

# Both produce agents with the same tools
api_agent = create_api_agent()
local_agent = create_local_agent()

print("Both agents use the same tools:")
print(f"  API agent tools: {[t.name for t in all_tools]}")
print(f"  Local agent tools: {[t.name for t in all_tools]}")

## 14. Tool Validation and Error Handling

Tools should handle errors gracefully. The agent loop continues even if a tool fails -- the error message is sent back to the LLM, which can then try a different approach:

In [ ]:
@tool
def safe_calculate_mean(values: list[float]) -> str:
    "Calculate the mean of a list of numbers with error handling."

    try:
        if not values:
            return "Error: Cannot calculate mean of an empty list."
        result = float(np.mean(values))
        return f"The mean is {result:.4f}"
    except TypeError:
        return "Error: All values must be numbers."
    except Exception as e:
        return f"Error calculating mean: {str(e)}"


@tool
def safe_calculate_std(values: list[float]) -> str:
    "Calculate the standard deviation with error handling."

    try:
        if len(values) < 2:
            return "Error: Standard deviation requires at least 2 values."
        result = float(np.std(values, ddof=1))
        return f"The sample standard deviation is {result:.4f}"
    except Exception as e:
        return f"Error: {str(e)}"


# Test error handling
print("Testing error handling:")
print(safe_calculate_mean.invoke({"values": []}))
print(safe_calculate_mean.invoke({"values": [10, 20, 30]}))
print(safe_calculate_std.invoke({"values": [42]}))
print(safe_calculate_std.invoke({"values": [10, 20, 30, 40, 50]}))

## 15. Responsible AI and Security

Tool-calling agents introduce **new security concerns** that do not exist with simple chains.

### Key Risks

```mermaid
graph TD
    A[Tool-Using Agent] --> B[Tool Permissions]
    A --> C[Code Execution]
    A --> D[Prompt Injection]
    A --> E[Data Privacy]
    
    B --> B1[Which tools can the agent access?]
    C --> C1[Can it run arbitrary code?]
    D --> D1[Can user input trick the agent?]
    E --> E1[Does data leave the machine?]
```

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Tool permissions** | Agent may call tools you didn't intend | Only pass tools the agent needs |
| **Arbitrary code execution** | Never let an agent run `eval()` or `exec()` | Use safe, validated tool functions |
| **Prompt injection** | Malicious input tricks the agent into misuse | Validate inputs inside tools |
| **Untrusted input** | Tool receives malformed or malicious data | Sanitize and validate all inputs |
| **Excessive autonomy** | Agent loops infinitely or takes unintended actions | Set `max_iterations` in AgentExecutor |
| **Data privacy** | API-based tools send data to external servers | Use local Ollama for sensitive data |
| **Output handling** | Agent output may contain sensitive information | Log and review agent responses |

In [ ]:
# Example: Setting max iterations to prevent infinite loops
safe_executor = AgentExecutor(
    agent=agent,
    tools=[calculate_mean, calculate_median, calculate_std],
    verbose=True,
    max_iterations=5,    # Stop after 5 tool calls
    handle_parsing_errors=True,  # Gracefully handle parsing issues
)

print("Safe executor created with:")
print("   - max_iterations: 5")
print("   - handle_parsing_errors: True")

In [ ]:
# Example: Input validation inside a tool
@tool
def validated_mean(values: list[float]) -> str:
    "Calculate the mean with strict input validation."

    # Validation
    if not isinstance(values, list):
        return "Error: Input must be a list of numbers."
    if len(values) == 0:
        return "Error: List cannot be empty."
    if len(values) > 10000:
        return "Error: List too large (max 10,000 elements)."
    if not all(isinstance(v, (int, float)) for v in values):
        return "Error: All elements must be numbers."
    
    result = float(np.mean(values))
    return f"Mean of {len(values)} values: {result:.4f}"

# Test validation
print(validated_mean.invoke({"values": [1, 2, 3]}))
print(validated_mean.invoke({"values": []}))
print(validated_mean.invoke({"values": list(range(10001))}))

## 16. When to Use Chains vs Agents

### Use a **Chain** when:

| Scenario | Example |
|----------|---------|
| Steps are known in advance | RAG pipeline: load -> split -> embed -> retrieve -> answer |
| You need predictable performance | Same input -> same output |
| Latency matters | One LLM call vs. multiple |
| Cost control is critical | Each call costs money |
| The workflow is simple | Prompt -> Model -> Parse |

### Use an **Agent** when:

| Scenario | Example |
|----------|---------|
| Tasks vary widely | "Calculate X" vs "Summarize Y" vs "Find Z" |
| You need tool selection | Different tools for different questions |
| Problem requires iteration | Multi-step reasoning |
| You want natural interaction | User asks open-ended questions |
| The workflow is dynamic | Steps depend on intermediate results |

```mermaid
graph LR
    A[What do you need?] --> B{Known workflow?}
    B -->|Yes| C[Use a Chain]
    B -->|No| D{Multiple tools?}
    D -->|Yes| E[Use an Agent]
    D -->|No| F[Use a Chain with prompt]
```

## Exercises

### Exercise 1: Create a Text Analysis Tool

Create a tool called `text_statistics` that takes a string and returns:
- Word count
- Character count  
- Average word length
- Number of unique words

Test it with the agent.

In [ ]:
@tool
def text_statistics(text: str) -> str:
    "Analyze text and return statistical information."

    words = text.split()
    word_count = len(words)
    char_count = len(text)
    avg_word_length = sum(len(w) for w in words) / max(word_count, 1)
    unique_words = len(set(w.lower() for w in words))
    
    return (f"Word count: {word_count}\n"
            f"Character count: {char_count}\n"
            f"Average word length: {avg_word_length:.1f}\n"
            f"Unique words: {unique_words}")


# Test it directly
print(text_statistics.invoke({"text": "Data science is the future of technology and innovation"}))

# Now add it to an agent and test!
# my_agent = create_tool_calling_agent(model, [text_statistics, ...], prompt)
# result = my_agent.invoke(...)

### Exercise 2: Build a Unit Converter Agent

Create three tools:
- `celsius_to_fahrenheit(celsius: float) -> float`
- `kilometers_to_miles(km: float) -> float`  
- `kilograms_to_pounds(kg: float) -> float`

Then build an agent that can answer: *"If it is 25C outside, what is that in Fahrenheit? And how far is 10km in miles?"*

In [ ]:
@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    "Convert Celsius to Fahrenheit."
    return celsius * 9/5 + 32

@tool
def kilometers_to_miles(km: float) -> float:
    "Convert kilometers to miles."
    return km * 0.621371

@tool
def kilograms_to_pounds(kg: float) -> float:
    "Convert kilograms to pounds."
    return kg * 2.20462

# Test directly
print(f"25C = {celsius_to_fahrenheit.invoke({'celsius': 25.0})}F")
print(f"10 km = {kilometers_to_miles.invoke({'km': 10.0})} miles")

# Build an agent and test multi-tool usage!
# agent = create_tool_calling_agent(model, [...], prompt)

### Exercise 3: Compare Agent vs Chain for Multi-Step Tasks

Create a chain that takes a list of numbers and:
1. Calculates the mean
2. Calculates the standard deviation
3. Formats the result

Then create an agent with the same tools. Compare:
- Number of LLM calls
- Latency
- Cost
- Flexibility

Write your observations below:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Chain approach
chain_prompt = ChatPromptTemplate.from_template(
    "Calculate the mean and standard deviation of these numbers: {numbers}"
)

chain = chain_prompt | model | StrOutputParser()

# Test the chain
result = chain.invoke({"numbers": "10, 20, 30, 40, 50"})
print("Chain result:", result)

# Compare with agent
# ... your comparison code here ...

## Challenges

### Challenge 1: Multi-Tool Data Analysis Agent

Build an agent with **all** of the following tools:
1. `calculate_mean`
2. `calculate_median`
3. `calculate_std`
4. `calculate_correlation`
5. `dataset_summary`
6. `text_statistics` (from Exercise 1)

Test it with a real CSV dataset (e.g., the Iris dataset from scikit-learn). Ask it at least 5 different questions and verify all answers are correct.

In [ ]:
# YOUR CODE HERE
# Load a real dataset
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
csv_data = iris.data.to_csv(index=False)

print("Iris dataset:")
print(iris.data.head())
print(f"\nShape: {iris.data.shape}")

### Challenge 2: Agent with Error Recovery

Build an agent that gracefully handles these edge cases:
- Empty dataset
- Non-numeric data
- Mismatched array lengths
- Very large numbers

Test each case and document how the agent recovers.

## Agent Design Exercise

Design an agent for a **Data Science Tutor** that helps students with:

1. Explaining concepts (use an LLM without tools)
2. Calculating statistics (use calculation tools)
3. Generating code examples (use the LLM)
4. Quizzing the student (use the LLM)

**Your task:** Write a design document with:
- List of tools needed
- System prompt for the agent
- 10 test questions that each require different tools
- Expected agent behavior for each question

In [ ]:
# YOUR DESIGN HERE
# Example starter:

ds_tutor_tools = [
    # What tools does your tutor need?
]

ds_tutor_prompt = "You are a Data Science Tutor..."

# Build and test your agent

## Troubleshooting

| Problem | Cause | Solution |
|---------|-------|----------|
| Agent selects no tools | Question too vague | Make the question more specific |
| `tool_calls` is empty | Model didn't recognize tool need | Improve tool descriptions in docstrings |
| Tool raised an error | Bad input to tool | Add input validation in the tool |
| Agent loops infinitely | Model keeps requesting tools | Set `max_iterations` on executor |
| Agent ignores tools | Model too generic | Strengthen the system prompt |
| Ollama tool calling fails | Model doesn't support tools | Use `llama3.2` or `qwen2.5` |
| ImportError: create_agent | Old langchain version | `pip install --upgrade langchain` |
| Schema validation error | Wrong argument types | Check type hints match tool usage |

### Debugging Tips

```python
# 1. Test tools directly (bypass the LLM)
result = my_tool.invoke({"arg": "value"})

# 2. Inspect tool calls from the model
response = model_with_tools.invoke(messages)
print(response.tool_calls)  # See what the model wants

# 3. Use verbose mode
executor = AgentExecutor(..., verbose=True)

# 4. Check available tools
for t in tools:
    print(f"{t.name}: {t.description[:50]}...")
```

## Key Takeaways

| Concept | Summary |
|---------|---------|
| **Tool** | A Python function the LLM can request to call |
| **`@tool` decorator** | Turns any function into a LangChain tool |
| **`bind_tools()`** | Attaches tool schemas to a model |
| **`tool_calls`** | Attribute on `AIMessage` showing requested tools |
| **Agent** | An LLM in a loop: think -> act -> observe -> repeat |
| **`create_tool_calling_agent`** | Classic agent builder (requires `AgentExecutor`) |
| **`create_agent`** | LangChain 1.0 simpler agent builder |
| **`AgentExecutor`** | Runs the agent loop (tool call -> execute -> feed back) |
| **`max_iterations`** | Safety limit to prevent infinite loops |
| **Error handling** | Tools should validate inputs and handle errors gracefully |
| **Security** | Limit tool scope, validate inputs, control autonomy |

### The Big Picture

```mermaid
graph TD
    subgraph Chain["Chains -- Fixed Workflows"]
        C1[Notebook 03: LCEL Chains]
        C2[Notebook 05: RAG Chains]
    end
    
    subgraph Agent["Agents -- Dynamic Workflows"]
        A1[Tools @tool]
        A2[Tool Calling bind_tools]
        A3[Agent Loop create_agent]
    end
    
    C1 --> A1
    C2 --> A1
    A1 --> A2
    A2 --> A3
    
    style Chain fill:#e3f2fd
    style Agent fill:#fff3e0
```

## Next Steps

In the final notebook **07_Advanced_LangChain_Project**, you will:

- Combine **everything** you've learned: chains, agents, RAG, tools
- Build a complete Data Science assistant
- Implement memory for multi-turn conversations
- Deploy a working prototype

> *You now understand the three pillars of LangChain: Chains for fixed workflows, RAG for knowledge-grounded generation, and Agents for dynamic tool use. The final notebook brings them all together.*